# 14 · Extreme-Power overlay — heterogeneous beams + RR + QED diagnostics

Compare three regimes:
1. Single-beam SEL CHF (a₀ = 130, λ = 0.8 μm) — the Timmis 2026 roadmap reference.
2. 12-beam dodecahedral SEL chf3d Phase C — homogeneous coherent multi-beam.
3. Heterogeneous 4×SEL + 4×ELI-NP cubic Extreme-Power — different λ per driver,
   Landau–Lifshitz radiation-reaction derate on, perturbative QED diagnostics on.

The point is the Schwinger-ratio progression and the regime where the
QED validity warning starts firing.

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from harmonyemissions.config import RunConfig
from harmonyemissions.qed import qed_diagnostics, schwinger_ratio, field_from_intensity
from harmonyemissions.runner import simulate_from_config
from harmonyemissions.units import a0_to_intensity

## 1. Single-beam SEL — Schwinger-ratio reference

In [ ]:
i_drive_w_per_cm2 = a0_to_intensity(130.0, 0.8)
i_drive_w_per_m2 = i_drive_w_per_cm2 * 1e4
ref = qed_diagnostics(
    intensity_w_per_m2=i_drive_w_per_m2 * 80.0,   # CHF gain Γ_total ≈ 80 from Timmis 2026
    omega_probe_rad_s=2.4e15,
    length_m=1e-6,
)
print('single-beam SEL+CHF reference:')
for k, v in ref.items():
    print(f'  {k}: {v}')

## 2. Phase C dodecahedral SEL chf3d (homogeneous)

In [ ]:
from harmonyemissions.config import load_config
cfg = load_config(Path('..') / 'configs' / 'chf_sel_array.yaml')
cfg.numerics.pipeline_grid = 32
cfg.numerics.chf_focal_volume_n = 8
result_phase_c = simulate_from_config(cfg)
print('chf3d Phase C, dodecahedral SEL — chf_gain:')
for k, v in result_phase_c.chf_gain.items():
    print(f'  {k}: {v:.3e}' if isinstance(v, float) else f'  {k}: {v}')

## 3. Extreme-Power overlay (heterogeneous SEL + ELI-NP)

In [ ]:
cfg_ep = load_config(Path('..') / 'configs' / 'extreme_power_combined.yaml')
cfg_ep.numerics.pipeline_grid = 32
cfg_ep.numerics.chf_focal_volume_n = 6
cfg_ep.extreme_power.omega_grid_points = 256
result_ep = simulate_from_config(cfg_ep)
print('Extreme-Power QED diagnostics:')
for k, v in result_ep.qed_diagnostics.items():
    print(f'  {k}: {v}')
print('\nRR per-beam derate summary:')
for i, m in enumerate(result_ep.radiation_reaction['per_beam']):
    print(f'  beam {i}: a₀={m["a0"]:.0f}, λ={m["wavelength_um"]:.2f}μm, derate={m["derate"]:.3f}')

## 4. Heterogeneous spectrum on the physical-frequency axis

When wavelengths differ across drivers the harmonic-order axis is no
longer shared, so the Result spectrum is emitted on `omega_rad_s`.

In [ ]:
spec = result_ep.spectrum
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(spec.coords['omega_rad_s'].values, np.maximum(spec.values, 1e-30))
ax.set_xlabel('ω [rad/s]')
ax.set_ylabel('on-axis coherent intensity (arb)')
ax.set_title('Extreme-Power coherent spectrum (8-beam cubic, SEL+ELI-NP)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()